In [0]:
# CHECK UNICITY IN DIMENSIONS by verifying equality
## IMPORTS AND CONFIGURATION
import pyspark.sql
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, sum as _sum, avg, count, countDistinct, round, coalesce, lit
)

dim_customers_data = spark.table("banking_gold.dim_customers")
total_without_unicidad = dim_customers_data.count()
total_unicidad = dim_customers_data.select(countDistinct(col("customer_sk"))).collect()[0][0] # Due to lazy evaluation, it needs collect() to execute, returning a list of rows. First row, first result.

# Only collect on small numbers, otherwise it downloads into driver memory causing an Out of Memory Error.

try:
    if total_without_unicidad != total_unicidad:
        raise Exception("Duplicate records found in customer_sk.")
except Exception as e:
    print(f"Warning, duplicate data found: {e}")
finally:
    print("End of Data Quality Check 1/5")

In [0]:
# CHECK UNICITY IN ACCOUNTS DIMENSION

dim_accounts_data = spark.table("banking_gold.dim_accounts")

total_without_unicidad = dim_accounts_data.count()
total_unicidad = dim_accounts_data.select(countDistinct(col("account_sk"))).collect()[0][0]

try:
    if total_without_unicidad != total_unicidad:
        raise Exception("Duplicate records found in account_sk.")
except Exception as e:
    print("Warning. Duplicate accounts found in the Accounts dimension.")
    print(f"There are {total_without_unicidad} accounts, and {total_unicidad} unique keys.")
finally:
    print("End of Data Quality Check 2/5")

In [0]:
# CHECK UNICITY IN TRANSACTIONS FACT TABLE

fact_tx = spark.table("banking_gold.fact_transactions")

total_tx = fact_tx.count()
unique_tx = fact_tx.select(countDistinct("transaction_id")).collect()[0][0]

try:
    if total_tx != unique_tx:
        raise Exception("Duplicate transaction records found.")
except Exception as e:
    print(f"WARNING. There are {total_tx - unique_tx} duplicate transactions.")
finally: 
    print("End of Data Quality Check 3/5")

In [0]:
# CHECK FOR GHOST TRANSACTIONS

from pyspark.sql.functions import col


gosht_trans = fact_tx.filter(
    (col("account_sk") == "UNKNOWN") | (col("customer_sk") == "UNKNOWN")
).count()

pct_gosht = float(f"{(gosht_trans / total_tx) * 100:.2f}")

try:
    if gosht_trans != 0:
        raise Exception("Ghost transactions detected.")
except Exception as e:
    print(f"WARNING: There are {gosht_trans} ghost transactions ({pct_gosht}% of total).")
finally:
    print("End of Data Quality Check 4/5")

In [0]:
# CHECK THAT ESSENTIAL DATA IS NOT NULL

from pyspark.sql.functions import col

null_trans = fact_tx.filter(
    col("amount").isNull() | 
    col("transaction_date_id").isNull() | 
    col("transaction_id").isNull()
).count()

pct_null = float(f"{(null_trans / total_tx) * 100:.2f}")

try:
    if null_trans != 0:
        raise Exception("Null values found in essential fields.")
except Exception as e:
    print(f"WARNING: There are {null_trans} transactions with null data ({pct_null}% of total).")
finally:
    print("End of Data Quality Check 5/5")